# notellm_magic v0.2.0 — Smoke Test

This notebook verifies the modernized extension after the `trio → anyio` migration and SDK upgrade.

**Prerequisites:**
- Install the package: `uv pip install -e .` (from repo root)
- Claude Code CLI installed and authenticated (either via `ANTHROPIC_API_KEY` or Claude Pro/Max subscription)

Run each section in order. Some cells require manual judgment (e.g., "did Claude respond?").

---
## 1. Extension Loading

Verify `%load_ext` works without import errors. You should see the welcome banner.

In [1]:
%load_ext notellm_magic



  Claude can execute shell commands, read/write/edit files, and access the web.
  Only use in trusted environments.

  Consider removing .claude/settings.local.json when done.


🚀 Claude Code Magic loaded!
Features:
  • Full agentic Claude Code execution
  • Cell-based code approval workflow
  • Real-time message streaming
  • Session state preservation
  • Conversation continuity across cells

Usage:
  %cc <instructions>       # Continue with additional instructions (one-line)
  %%cc <instructions>      # Continue with additional instructions (multi-line)
  %cc_new (or %ccn)        # Start fresh conversation
  %cc_cur (or %ccc)        # Like %cc, but replaces the prompt cell in-place
  %cc --help               # Show available options and usage information

Context management:
  %cc --import <file>       # Add a file to be included in initial conversation messages
  %cc --add-dir <dir>       # Add a directory to Claude's accessible directories
  %cc --mcp-config <file>   # Set path 

### 1b. Version Check

Confirm the version is `0.2.0`.

In [2]:
from notellm_magic.cc_jupyter import __version__
print(f"Version: {__version__}")
assert __version__ == "0.2.0", f"Expected 0.2.0, got {__version__}"

Version: 0.2.0


---
## 2. Help Display

Verify `--help` prints usage info and returns (no API call).

In [3]:
%cc --help


🚀 Claude Code Magic loaded!
Features:
  • Full agentic Claude Code execution
  • Cell-based code approval workflow
  • Real-time message streaming
  • Session state preservation
  • Conversation continuity across cells

Usage:
  %cc <instructions>       # Continue with additional instructions (one-line)
  %%cc <instructions>      # Continue with additional instructions (multi-line)
  %cc_new (or %ccn)        # Start fresh conversation
  %cc_cur (or %ccc)        # Like %cc, but replaces the prompt cell in-place
  %cc --help               # Show available options and usage information

Context management:
  %cc --import <file>       # Add a file to be included in initial conversation messages
  %cc --add-dir <dir>       # Add a directory to Claude's accessible directories
  %cc --mcp-config <file>   # Set path to a .mcp.json file containing MCP server configurations
  %cc --cells-to-load <num> # The number of cells to load into a new conversation (default: all for first %cc, none for %c

---
## 3. Configuration Options (no API calls)

These settings change internal state without making API calls.

In [4]:
# Should print confirmation of the new max_cells setting
%cc --max-cells 5

📝 Set max_cells from 3 to 5. Will apply to the next query.


In [5]:
# Should print confirmation of model selection
%cc --model opus

✅ Set model to opus. Will apply to the next query.


In [6]:
# Should print confirmation of cells-to-load setting
%cc --cells-to-load 3

✅ Will load up to 3 recent cell(s) when starting new conversations


In [7]:
# Should print confirmation that /tmp was added to accessible directories
# This tests the new `add_dirs` SDK feature (replaces old settings JSON hack)
%cc --add-dir /tmp

✅ Added /private/tmp to accessible directories. Will apply to the next query.


In [8]:
# Reset model to sonnet for the API tests below
%cc --model sonnet

✅ Set model to sonnet. Will apply to the next query.


---
## 4. Basic Query (API Call)

Send a simple prompt. Expect:
- Model name printed (`Claude model: ...`)
- A `CreateNotebookCell` tool call
- A new code cell inserted below
- Session ID printed

In [9]:
%cc Print "hello from notellm v0.2.0" using python


🤖 Claude wants to execute code
------------------------------------------------------------
📋 To approve: Run the cell below
➡️ To continue Claude agentically afterward: Run %cc

🧠 Claude model: claude-sonnet-4-6
⏺ CreateNotebookCell
📍 Claude Code Session ID: 4c9524e9-d68d-4aa9-91e6-1c56850a0733


In [ ]:
print("hello from notellm v0.2.0")

---
## 5. Session Continuity

Follow up on the previous query. This tests that the `session_id` is reused
and `anyio`-based streaming works across multiple turns.

Run the cell generated by step 4 first, then run the cell below.

In [10]:
%cc Now modify the code above to also print the current date and time

✅ Continuing Claude session with execution results...

🤖 Claude wants to execute code
------------------------------------------------------------
📋 To approve: Run the cell below
➡️ To continue Claude agentically afterward: Run %cc



💭 Claude: `★ Insight ─────────────────────────────────────`
Python's `datetime` module is part of the standard library — no imports from external packages needed. `datetime.now()` returns a local timestamp; for timezone-aware output, use `datetime.now(timezone.utc)`.
`─────────────────────────────────────────────────`

⏺ CreateNotebookCell


💭 Claude: `★ Insight ─────────────────────────────────────`
`strftime` format codes follow C's `time.h` conventions — `%Y` is 4-digit year, `%m` is zero-padded month, `%d` is zero-padded day, `%H:%M:%S` is 24-hour time. This is more readable than the default `datetime.__str__()` output which includes microseconds.
`─────────────────────────────────────────────────`

In [ ]:
from datetime import datetime

print("hello from notellm v0.2.0")
print(f"Current date and time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 6. New Conversation (`%cc_new`)

Start a fresh session with a new prompt. `%cc_new` resets the session ID and requires a prompt.
The session ID should be different from step 4.

In [11]:
%cc_new Print "fresh session" using python. No comments.


🤖 Claude wants to execute code
------------------------------------------------------------
📋 To approve: Run the cell below
➡️ To continue Claude agentically afterward: Run %cc

🧠 Claude model: claude-sonnet-4-6
⏺ CreateNotebookCell
📍 Claude Code Session ID: b4abfc6c-579d-45a3-b1f1-7060a2ac8228


In [ ]:
print("fresh session")

---
## 7. Multi-line Magic (`%%cc`)

Test the cell magic form with multi-line instructions.

In [12]:
import math

def stats(numbers: list[float]) -> tuple[float, float]:
    n = len(numbers)
    mean = math.fsum(numbers) / n
    variance = math.fsum((x - mean) ** 2 for x in numbers) / n
    return mean, math.sqrt(variance)

stats([2, 4, 4, 4, 5, 5, 7, 9])

✅ Continuing Claude session with execution results...

🤖 Claude wants to execute code
------------------------------------------------------------
📋 To approve: Run the cell above
➡️ To continue Claude agentically afterward: Run %cc



💭 Claude: `★ Insight ─────────────────────────────────────`
`math.fsum()` is used instead of `sum()` for floating-point accumulation — it uses an exact summation algorithm that avoids catastrophic cancellation errors on large or mixed-magnitude lists.
`─────────────────────────────────────────────────`

⏺ CreateNotebookCell


💭 Claude: `★ Insight ─────────────────────────────────────`
This returns **population** standard deviation (divides by `n`), not sample std dev (which divides by `n-1`). The classic list `[2,4,4,4,5,5,7,9]` is a textbook example — its population std dev is exactly `2.0`, making it a good sanity check.
`─────────────────────────────────────────────────`

---
## 8. Variable Tracking

Create a variable, then ask Claude about it. The variable tracker should report
the new variable in the context sent to Claude.

In [11]:
test_data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [ ]:
%cc What is the sum of test_data? Compute it in a new cell.

---
## 9. Import File

Test importing a file for context. Create a temp file first, then import it.

In [ ]:
import tempfile, os

# Create a temp file with some content
tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False)
tmp.write('def greet(name):\n    return f"Hello, {name}!"\n')
tmp.close()
print(f"Created temp file: {tmp.name}")

In [ ]:
# Import the file — should print success message
%cc --import {tmp.name}

In [ ]:
os.unlink(tmp.name)
print("Cleaned up temp file")

---
## 11. Interrupt Test (Manual)

Run the cell below, then press **Kernel > Interrupt** (or `Ctrl+C` in terminal)
while Claude is responding. You should see `Query interrupted by user`.

This tests the `anyio.create_task_group` interrupt handling.

---
## 10. Replace Current Cell (`%cc_cur`)

`%cc_cur` (alias `%ccc`) works like `%cc` but replaces the prompt cell in-place
instead of inserting a new cell below. The cell containing `%cc_cur` should be
overwritten with Claude's generated code.

In [ ]:
%cc_cur Write a one-liner that prints the sum of 1 to 100

In [ ]:
# Give Claude a task that takes a while so you have time to interrupt
%cc Write a detailed essay about the history of computing from 1940 to 2020

---
## Results Checklist

| # | Test | Expected | Pass? |
|---|------|----------|-------|
| 1 | `%load_ext` | Welcome banner, no errors | |
| 1b | Version | `0.2.0` | |
| 2 | `--help` | Usage text with `%cc_cur` listed | |
| 3 | Config options | Confirmation messages | |
| 4 | Basic query | Cell created below, session ID shown | |
| 5 | Continuation | Same session, new cell below | |
| 6 | `%cc_new` | New session ID | |
| 7 | `%%cc` multi-line | Function created | |
| 8 | Variable tracking | Claude knows `test_data` | |
| 9 | `--import` | File added to import list | |
| 10 | `%cc_cur` | Prompt cell replaced (not inserted below) | |
| 11 | Interrupt | "Query interrupted" message | |